In [ ]:
import torch
import json
import re
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import PeftModel
from datasets import load_dataset

In [ ]:
SPIDER_TABLES = "/mnt/storage_C1/igorzwirtes/poster_ic/spider/tables.json"
FILE_PATH = "/mnt/storage_C1/igorzwirtes/poster_ic/predictions/predictions_base_2.json"

In [ ]:
model_path = "/mnt/storage_C1/igorzwirtes/poster_ic/qwen2.5coder"
adapter_path = "/mnt/storage_C1/igorzwirtes/poster_ic/lora_weights/r64q4a128_2"

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(model_path)
tokenizer.pad_token = tokenizer.eos_token

In [ ]:
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

base_model = AutoModelForCausalLM.from_pretrained(
    model_path,
    quantization_config=bnb_config,
    device_map="auto",
)

model = PeftModel.from_pretrained(base_model, adapter_path)

model.config.use_cache = True

model.eval();

In [ ]:
dataset = load_dataset("spider")
test_data = dataset["validation"]

In [ ]:
with open(SPIDER_TABLES) as f:
    tables_data = json.load(f)

# (mesma função do train.ipynb)
def format_schema_with_fk(db):
    col_names = db["column_names_original"]
    lines = []
    for i, table in enumerate(db["table_names_original"]):
        cols = [col[1] for col in col_names if col[0] == i]
        lines.append(f"  {table}({', '.join(cols)})")
    if db.get("foreign_keys"):
        lines.append("  Foreign keys:")
        for fk in db["foreign_keys"]:
            c1 = col_names[fk[0]]
            c2 = col_names[fk[1]]
            t1 = db["table_names_original"][c1[0]]
            t2 = db["table_names_original"][c2[0]]
            lines.append(f"    {t1}.{c1[1]} → {t2}.{c2[1]}")
    return "\n".join(lines)

schema_index = {db["db_id"]: format_schema_with_fk(db) for db in tables_data}

def build_prompt(example):
    schema = schema_index.get(example["db_id"], "")
    messages = [
        {"role": "system", "content": (
            "You are a Text-to-SQL translator.\n"
            "Generate a valid SQLite SQL query.\n"
            "Use only tables and columns from the schema.\n"
            "Output only the SQL query.\n"
            "Do not explain.\n"
            "Do not use markdown."
        )},
        {"role": "user", "content": (
            f"Database: {example['db_id']}\n"
            f"Schema:\n{schema}\n\n"
            f"Question: {example['question']}"
        )}
    ]
    return tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)

In [ ]:
def extract_sql(text):
    # bloco markdown
    match = re.search(
        r"```(?:sql)?\s*(.*?)```",
        text,
        re.DOTALL | re.IGNORECASE
    )

    if match:
        sql = match.group(1).strip()
    else:
        # pega primeira query SQL
        match = re.search(
            r"(SELECT|INSERT|UPDATE|DELETE|WITH)\b.*?;",
            text,
            re.DOTALL | re.IGNORECASE
        )

        if match:
            sql = match.group(0).strip()
        else:
            sql = text.strip()

    # remove comentários
    sql = re.sub(r"--.*", "", sql)

    # remove markdown sobrando
    sql = sql.replace("```", "").strip()

    return sql

In [ ]:
predictions = []

for example in test_data:
    prompt = build_prompt(example)
    inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=2048).to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=256,
            do_sample=False,
            num_beams=1,
            early_stopping=False,
            pad_token_id=tokenizer.eos_token_id,
            eos_token_id=[
                tokenizer.eos_token_id,
                tokenizer.convert_tokens_to_ids("<|im_end|>"),
            ],
        )

    # Decodifica só os tokens novos
    generated = outputs[0][inputs["input_ids"].shape[1]:]
    pred_sql = extract_sql(
        tokenizer.decode(
            generated,
            skip_special_tokens=True,
            clean_up_tokenization_spaces=False
        )
    )
    pred_sql = pred_sql.split(";")[0]

    predictions.append({
        "db_id": example["db_id"],
        "question": example["question"],
        "gold": example["query"],
        "predicted": pred_sql,
    })
    print(f"Q: {example['question']}\nGold: {example['query']}\nPred: {pred_sql}\n---")

with open(FILE_PATH, "w") as f:
    json.dump(predictions, f, indent=2)

print(f"\nPredições salvas: {len(predictions)} exemplos")